In [ ]:
import numpy as np
import random
from mesa import Model
from mesa.space import SingleGrid
from mesa.datacollection import DataCollector
# importamos el fuego
from models.fuego import crear_matriz_fuego, advance_fire
from models.pois import crear_matriz_pois

In [ ]:
class FlashPoint (Model):
    def __init__(self, width = 8, height = 6, poi_posiciones = None, **kwargs):
        super().__init__()
        self.width = width
        self.height = height
        self.grid = SingleGrid(width, height, torus = False)

        self.fuego = crear_matriz_fuego(height, width)
        self.pois = crear_matriz_pois(height, width, poi_posiciones or [])
        self.turno = 0
        self.datacollector = DataCollector(model_reporters = {"Grid": self._get_grid})
    def _get_grid(self):
        grid  = self.fuego.copy()
        for content, (x,y) in self.grid.coord_iter():
            if content is not None:
                grid[y][x]=7
        return grid

    def step(self):
        advance_fire(self.fuego, self.height, self.width)
        self.turno += 1
        self.datacollector.collect(self)

    def to_dict(self):
        return {
            "turno": self.turno,
            "width": self.width,
            "height": self.height,
            "fuego": self.fuego.flatten().tolist(),
            "pois": self.pois.flatten().tolist(),
            "agentes": [
                {"id": a.unique_id, "x": a.pos[0], "y": a.pos[1]}
                for a in self.agents
            ] if hasattr(self, "agents") else []
            
        }